# Jupyter und API-Anfragen (am Beispiel vom Bibliothekssystem FOLIO)

## Anmeldung gegen den FOLIO-Okapi-Client

Um API-Anfragen in FOLIO durchzuführen, muss auf den sog. Okapi-Client als Hilfmittel zurückgegriffen werden. <br>
In dieser Umgebung ist der [Okapi-Client](https://gitlab.bib-bvb.de/folio-public/okapiclient/-/tree/master?ref_type=heads) bereits installiert.

In [50]:
'''DIESE ZELLE VOR ALLEN ANDEREN AUSFÜHREN'''

import json                                                     # FOLIO liefert und erwartet JSON-Daten
from getpass import getpass                                     # Standardmäßig bei Python dabei
from OkapiAPIClient.ApiClient import ApiClient                  # Import OkapiClient (muss zuerst installiert werden | https://gitlab.bib-bvb.de/folio-public/okapiclient)

helper = ApiClient(                                             
    url="https://folio-snapshot-okapi.dev.folio.org",           # Basis-URL an eigenen Mandanten anpassen
    username="API_Test",                                        # FOLIO-Username mit gewünschten Rechten
    password=getpass(),                                         # Zur verdeckten Eingabe das Passworts (kann aber im Zweifel auch in Klartext hingeschrieben werden)
    tenant="diku",                                              # Tenant-ID an eigenen Mandanten anpassen
    verbose=True                                                # Für klarere Fehlermeldungen wichtig
)

### API-Dokumentation

Eine Beschreibung der FOLIO-Endpunkte findet sich hier: https://dev.folio.org/reference/api/

## Beispiele und Übungen

### GET-Anfragen

#### Grundlagen

In [ ]:
# Abfrage aller FOLIO-Nutzeraccounts (standardmäßig werden nur 10 Standorte ausgegeben)

response = helper.makeAPICall(
    path = '/users'                 # Endpoint, der an die Basis-URL von oben angefügt wird
).json()                            # API-Antwort wird als JSON ausgegeben

response

In [ ]:
# Beispiel Python List-Control

response = helper.makeAPICall(
    path = '/users'                             # Gleiche Anfrage wie oben
).json()

# Mit List-Control lässt sich zum Beispiel gezielt die Mailadresse des fünften Nutzers in der Liste ausgeben
response["users"][4]["personal"]["email"]       # → Hangelt sich durch die gesamte JSON-Struktur hindurch, die aus Dictionaries und Listen besteht

In [ ]:
# f-String

response = helper.makeAPICall(
    path = '/users'
).json()

# ID des ersten Nutzerdatensatzes wird in einer Variable gespichert
user_id = response["users"][0]["id"]

# GET-Anfrage für einzelnen Nutzer hat einen eigenen Endpunkt
single_user = helper.makeAPICall(
    path = f'/users/{user_id}'      # Der f-String erlaubt es Variablen innerhalb eines Strings zu verwenden
).json()

single_user

#### Übung 1

1. Führe eine GET-Abfrage für den Endpoint '/inventory/items' durch.
2. Wie viele Exemplare sind insgesamt vorhanden?
3. Gib den Titel des dritten Mediums in der Liste aus.

In [ ]:
# Hier die Lösung für Übung 1 eintragen:

 





#### List Comprehension und Schleifen

In [ ]:
# Beispiel: Liste mit allen Exemplar-IDs, die jemals ausgeliehen wurden

item_ids = []       # Leere Liste wird initialisiert. Hier sollen am Ende die Item-IDs landen

# Gewöhnliche GET-Anfrage
response = helper.makeAPICall(
    path = '/loan-storage/loans'
).json()

for item in response["loans"]:          # For-Schleife. Jedes Element aus response["loans"] wird ausgelesen und mit dem Variablennamen "item" versehen.
    item_ids.append(item["itemId"])     # Aus der einzelnen Ausleihe wird die ID des betroffenen Items ausgelesen und zur Liste hinzugefügt.

item_ids

#### Übung 2

Erstelle eine Liste, die den Titel jedes bereits ausgeliehenen Mediums aufführt. <br>
Tipp: Es bietet sich der Endpunkt '/inventory/items/{uuid}' an.

In [ ]:
# Hier die Lösung für Übung 2 eintragen:







In [ ]:
# SPÄTERE ERGÄNZUNG (Zelle zunächst unsichtbar)

title_list = dict()

for item in item_ids:    
    response = helper.makeAPICall(
        path = f'/inventory/items/{item}'
    ).json()

    title_list[item] = response["title"]

title_list

{'23fdb0bc-ab58-442a-b326-577a96204487': 'Temeraire',
 '459afaba-5b39-468d-9072-eb1685e0ddf4': 'The Girl on the Train',
 'bb5a6689-c008-4c96-8f8f-b666850ee12d': 'Interesting Times',
 '23f2c8e1-bd5d-4f27-9398-a688c998808a': 'Nod',
 '1b6d3338-186e-4e35-9e75-1b886b0da53e': "Bridget Jones's Baby: the diaries"}

#### CQL-Abfragen

Anregungen für CQL-Suchen sind hier: https://folio-org.atlassian.net/wiki/spaces/FOLIOtips/pages/5671792/Search+-+using+Elasticsearch+or+OpenSearch <br>
CQL-Suchabfragen unterscheiden sich aber teilweise von Endpunkt zu Endpunkt

In [ ]:
# Abfrage der Nutzer-Accounts, die zur Nutzergruppe "graduate" gehören

graduate = "ad0bc554-d5bc-463c-85d1-5562127ae91b"

response = helper.makeAPICall(
    path = f'/users?query=patronGroup={graduate} and active="True"&limit=50'                # Modifikatoren zum Endpoint werden über ? vom Pfad abgetrennt
).json()

response

In [ ]:
# Abfrage für alle aktuell ausgeliehenen Exemplare

response = helper.makeAPICall(
    path = '/inventory/items?query=status.name="Checked out"'
).json()

response["items"]

#### Übung 3

Ermittle die Anzahl der Exemplare, die von der Nutzergruppe "graduate" bisher entliehen wurde. <br>
Tipp: Es bietet sich der Endpunkt '/loan-storage/loans' an.

In [ ]:
# Hier die Lösung für Übung 3 eintragen:







### POST und PUT

#### POST - Neue Datensätze erstellen

In [ ]:
# Neuen Nutzerdatensatz erstellen

# Daten für den Upload werden als JSON in der Variable "user" definiert
user = {
  "active" : True,
  "type" : "patron",
  "patronGroup" : "ad0bc554-d5bc-463c-85d1-5562127ae91b",
  "personal" : {
    "firstName" : "Daniel",
    "lastName" : "Diplodocus",
    "email" : "nature.calls@wild.test",
    "addresses" : [ ],
    "preferredContactTypeId" : "002"
  }
}

upload_user = helper.makeAPICall(
    path = f'/users',           # Gleicher Endpunkt wie bei der GET-Anfrage oben
    method = 'post',            # Im Gegensatz zu GET muss hier die Methode definiert werden
    data = json.dumps(user)     # Greift auf Variable "user" zurück
)

upload_user

#### PUT - Datensätze verändern

In [ ]:
# Ausleihtyp bei einem bestimmten Exemplar verändern

item_id = "30000000-0000-4000-8000-000000000008"

# Zunächst wird das Exemplar per GET-Anfrage abgeholt
get_item = helper.makeAPICall(
    path = f'inventory/items/{item_id}'
).json()

# Das Feld "permanentLoanType.id" wird explizit angesteuert und verändert
get_item["permanentLoanType"]["id"] = "2e48e713-17f3-4c13-a9f8-23845bb210a4" # <-- LoanTypeId für "Reading Room"

# Anschließend wird das veränderte Exemplar wieder hochgeladen
put_item = helper.makeAPICall(
    path = f'inventory/items/{item_id}',
    method = "put",                 # PUT-Methode
    data = json.dumps(get_item)     # Variable der veränderten Antwort aus der GET-Anfrage
)

put_item

#### Übung 4

Ändere den Titel eines beliebigen Werkes in der Katalog-App. <br>
Tipp: Es bietet sich der Endpunkt '/inventory/instances/{uuid}' an.

In [ ]:
# Hier die Lösung für Übung 4 eintragen:







### Pandas

Pandas ist eine Python-Bibliothek zur Datenverarbeitung, die es z.B. erlaubt sehr komfortabel CSV-Dateien zu erzeugen

In [80]:
# CSV-Datei mit allen inaktiven Nutzerdaten erstellen

import pandas

# Name der CSV-Datei, die ausgegeben werden soll, wird definiert
filename = "Nutzerdaten.csv"

# Abfrage der Nutzerdaten
response = helper.makeAPICall(
    path = '/users?query=active="False"&limit=1000'
).json()

# Pandas kann einen sog. DataFrame erstellen - was einer Tabelle entspricht
user_data = pandas.DataFrame(response["users"])
user_data.to_csv(filename, index=False, encoding="utf-8") # CSV wird generiert


### Beispiele für Abfragen aus der Praxis 

!!Achtung!! <br>
Die folgenden Abfragen sind eher als Anschauungsbeispiele zu verstehen und müssen meistens an die eigene FOLIO-Umgebung angepasst werden

In [ ]:
# CSV-Datei mit Titeldaten aus einem bestimmten Fonds

import pandas

fund_id = "8f89f4e1-6d4b-41e7-a11b-5fbc60459ba8"
results = []

order_lines = helper.makeAPICall(
    path=f'/orders/order-lines?query=(fundDistribution =/@fundId ({fund_id}))&limit=1000'
).json()

for order in order_lines["poLines"]:

    results.append({
        "Instanz-ID": order.get("instanceId", ""),
        "Titel": order.get("titleOrPackage", ""),
        "Verlag": order.get("publisher", ""),
        "Erscheinungsdatum": order.get("publicationDate", ""),
        "Standort": order.get("searchLocationIds", ""),
        "Bestellposten": order.get("poLineNumber", "")
    })

# DataFrame erst NACH der Schleife bauen
table = pandas.DataFrame(results)

filename = "Query_Title-Fund"
table.to_csv(filename, index=False, encoding="utf-8")

#### Beispiel: Änderung Fernleihrelevanz

In [ ]:
# "f75727f2-6d4b-4535-8c93-af95ad814234" = bedingt fernleihrelevant
# "daa11288-8049-4f8b-89b0-06d56e11bb53" = fernleihrelevant
# "bdce069f-57ce-4904-8fa5-031d27ba3b98" = kopierbar
# "4d89179a-0021-4b73-b3d8-a6eae7dd3fa2" = Negativnachweis
# "24097d7a-5a4e-439b-9f12-94f791e916df" = nicht fernleihrelevant

In [ ]:
# Location-ID herausfinden

response = helper.makeAPICall(
    path = 'locations?limit=3000&query=code=="00/ZX 3201" sortby name'
).json()

response

In [ ]:
# Fernleihrichtlinien nur bei lokalen Holdings ändern inkl. Limit und Offset und Abspeichern einer Backup-Datei vor dem PUT-Call

import os
import copy

archiv_ordner = "Archiv_Änderungen"
os.makedirs(archiv_ordner, exist_ok=True)

limit = 5000
offset = 0
effective_location_id = "faac43e7-cd89-4ec0-ad13-923dfaea3034" # ← Hier Location hinterlegen

filename = f"holdings_storage_{effective_location_id}.json"
filepath = os.path.join(archiv_ordner, filename)

all_records = []

while True:
    response = helper.makeAPICall(
        path =f'holdings-storage/holdings?query=effectiveLocationId=={effective_location_id} and sourceId=="d8cc7176-f53d-5c87-971a-b29511ba360b"&limit={limit}&offset={offset}'
    ).json()

    data = response.get("holdingsRecords", [])

    if not data:
        break

    # Originaldaten sichern
    all_records.extend(copy.deepcopy(data))

    for record in data:

        record.pop("metadata", None)
        record.pop("holdingsItems", None)
        record.pop("bareHoldingsItems", None)
        record["illPolicyId"] = "bdce069f-57ce-4904-8fa5-031d27ba3b98" # ← Hier Fernleihrichtlinie hinterlegen
        
        response = helper.makeAPICall(
            path =f'holdings-storage/holdings/{record["id"]}',
            data = json.dumps(record),
            method = "put"
        )

        print(response)

    offset = offset + limit

# Archivdatei schreiben
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"Originaldaten archiviert unter: {filepath}")